In [15]:
import pandas as pd

In [16]:
df = pd.read_json("data.json", lines=True)

In [17]:
df.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,g5E1V500RBftRv_XmkBkcg,The Corner Bar,5506 S Meridian,Indianapolis,IN,46217,39.685943,-86.158923,4.0,39,1,"{'RestaurantsPriceRange2': '1', 'RestaurantsGo...","Restaurants, Nightlife, Sports Bars, American ...","{'Monday': '0:0-0:0', 'Tuesday': '11:0-0:0', '..."
1,183RrwDm8mmdKCX1E9dUVQ,K & A Bagel Cafe,1426 NJ-70,Cherry Hill,NJ,08034,39.911206,-74.992019,4.5,177,1,"{'BusinessAcceptsCreditCards': 'True', 'Ambien...","Breakfast & Brunch, Restaurants, Sandwiches, F...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-13:0', '..."
2,ofEy7YsTR6JMVYRZljHJZQ,America Choice RV - Zephyrhills,3334 Paul S Buchman Hwy,Zephyrhills,FL,33540,28.205882,-82.172990,1.0,5,0,None,"Automotive, RV Dealers","{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
3,NQn2fIomrrMXXUxZXH99YA,Rox Cafe,5134 Rochelle Ave,Philadelphia,PA,19128,40.017768,-75.211206,2.5,12,1,"{'OutdoorSeating': 'True', 'WiFi': 'u'free'', ...","Food, Restaurants, Breakfast & Brunch, Sandwic...","{'Monday': '7:0-19:0', 'Tuesday': '7:0-19:0', ..."
4,fHhqyONubL3XD6FN7BzSug,Krystal,449 W Main St,Hendersonville,TN,37075,36.301940,-86.630942,2.0,15,1,"{'Ambience': '{'romantic': False, 'intimate': ...","Burgers, Restaurants, Fast Food","{'Monday': '6:0-0:0', 'Tuesday': '6:0-0:0', 'W..."


In [18]:
df.shape

(2716, 14)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2716 entries, 0 to 2715
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   2716 non-null   object 
 1   name          2716 non-null   object 
 2   address       2716 non-null   object 
 3   city          2716 non-null   object 
 4   state         2716 non-null   object 
 5   postal_code   2716 non-null   object 
 6   latitude      2716 non-null   float64
 7   longitude     2716 non-null   float64
 8   stars         2716 non-null   float64
 9   review_count  2716 non-null   int64  
 10  is_open       2716 non-null   int64  
 11  attributes    2472 non-null   object 
 12  categories    2714 non-null   object 
 13  hours         2278 non-null   object 
dtypes: float64(3), int64(2), object(9)
memory usage: 297.2+ KB


In [20]:
import numpy as np
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [21]:
content_data = df[['business_id', 'name', 'categories', 'attributes', 'stars', 'review_count']].copy()

# Set type to string
content_data['categories'] = content_data['categories'].astype(str)
content_data['attributes'] = content_data['attributes'].astype(str)

# handle missing values
content_data['categories'].fillna('', inplace=True)
content_data['attributes'].fillna('', inplace=True)

# make a new column called 'content'
content_data['content'] = content_data['name'] + ' ' + content_data['categories'] + ' ' + content_data['attributes']


<ipython-input-21-693c0681972d>:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  content_data['categories'].fillna('', inplace=True)
<ipython-input-21-693c0681972d>:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [22]:
# TF-IDF to convert text into numerical vectors
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix =tfidf_vectorizer.fit_transform(content_data['content'])

# Calculate cosine similarity
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [23]:
def get_content_recommendations(name, cosine_sim=cosine_sim, content_data=content_data, threshold=0.1):

    # get index of the restaurent
    idx = content_data.index[content_data['name'] == name].tolist()
    if not idx:
        print(f"No restuarents found with name '{name}'")
        return []

    idx = idx[0]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # filter out items with similarity below the threshold
    sim_scores = [(i, score) for i, score in sim_scores if score > threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # get top 10 similar restaurents
    top_similar_restaurents = sim_scores[1:11]

    # Get the indices and names of the top similar restaurents
    similar_indices = [i[0] for i in top_similar_restaurents]
    similar_restaurents = content_data.iloc[similar_indices][['name', 'review_count', 'stars']]
    return similar_restaurents

In [42]:
restaurent_name = "Domino's Pizza"
content_recommendations = get_content_recommendations(restaurent_name, threshold=0.5)

print(f"Content-Based Recommendations for {restaurent_name}:")
print(content_recommendations)

Content-Based Recommendations for Domino's Pizza:
                name  review_count  stars
573   Domino's Pizza             8    3.0
395   Domino's Pizza            22    2.0
1445  Domino's Pizza            56    1.5
2517  Domino's Pizza            19    2.0
1740  Domino's Pizza            28    2.0
1521   King Of Pizza            26    3.5
2209  Paradise Pizza            78    4.0
569    King of Pizza           141    3.5
390   Domino's Pizza            15    1.5
855        Pizza Hut            13    2.5


In [50]:
restaurent_name = "Old English Pizza"
content_recommendations = get_content_recommendations(restaurent_name, threshold=0.5)

print(f"Content-Based Recommendations for {restaurent_name}:")
print(content_recommendations)

Content-Based Recommendations for Old English Pizza:
                    name  review_count  stars
1521       King Of Pizza            26    3.5
2209      Paradise Pizza            78    4.0
2660          Pizza Guru           299    4.0
369        Pizza Suprema           111    4.0
2528         Sal's Pizza           150    3.5
677   Bloomingdale Pizza            87    3.5
1629     Chicago's Pizza            18    4.0
2278   Scotto Pizza Cafe            40    4.0
528    Pizza Time Saloon            26    3.5
2154   Papa John's Pizza            12    4.0


In [53]:
restaurent_name = "Sic Ink"
content_recommendations = get_content_recommendations(restaurent_name, threshold=0.5)

print(f"Content-Based Recommendations for {restaurent_name}:")
print(content_recommendations)

Content-Based Recommendations for Sic Ink:
                                         name  review_count  stars
1843   Under Your Skin Tattoo & Body Piercing            61    4.0
2533  Queen of Hearts Tattoos & Body Piercing            39    4.0
